# kaiming-uniform-init — ex1: build a Linear with Kaiming-uniform init + histogram visualization

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `kaiming-uniform-init`. Running the final beacon cell reports progress against the `Init: Kaiming uniform` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import math
import matplotlib.pyplot as plt

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Init: Kaiming uniform` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`kaiming-uniform-init`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "kaiming-uniform-init"
DD_SUBTOPIC = "Init: Kaiming uniform"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Kaiming-uniform init — quick refresher

PyTorch's default `nn.Linear` / `nn.Conv2d` init is Kaiming-uniform with `a=sqrt(5)`, which simplifies to:

```python
bound = 1 / math.sqrt(fan_in)
weight.uniform_(-bound, +bound)
```

where `fan_in = in_features` for Linear, `in_channels * kH * kW` for Conv2d. The general He/Kaiming formula `Uniform(-sqrt(6 / fan_in), +sqrt(6 / fan_in))` corresponds to `a=0` (pure ReLU) — PyTorch uses the `a=sqrt(5)` variant for historical compatibility, which gives the cleaner `1/sqrt(fan_in)` bound used above.

**Why uniform, not normal.** Both give variance `~2 / fan_in` for the ReLU-flavored variant, but uniform has bounded support — no extreme outliers at init, which empirically helps very deep networks converge.

**Why `fan_in` and not `fan_out`.** Forward-pass variance preservation. If you also care about backward-pass variance, average them (`fan_avg`). PyTorch's default uses `fan_in` because for most architectures forward stability matters more than backward.

**The bias.** PyTorch initializes bias from the same `Uniform(-bound, +bound)` distribution as the weight (using the weight's fan_in). `nn.init.kaiming_uniform_` only touches the weight; you handle bias yourself with a matching `.uniform_(-bound, +bound)`.

### Exercise 1 — build a Linear with Kaiming-uniform init + histogram visualization

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Build a Linear layer whose weight is initialized from Uniform(-1/sqrt(fan_in), +1/sqrt(fan_in)) and confirm the empirical distribution matches the theoretical bounds via a histogram.
> Keywords: kaiming, init, uniform, fan_in, histogram
> ```

**KCs targeted:** `kaiming-uniform-bound-formula`, `init-weight-uniform-in-place`

Implement `ex1_kaiming_linear(in_features, out_features)` to build a Linear-style Module whose weight is initialized using PyTorch's default Kaiming-uniform formula:

1. Class `KaimingLinear(t.nn.Module)`:
   - `__init__(self, in_features, out_features)`:
     - `super().__init__()`.
     - Store `self.in_features` and `self.out_features`.
     - Compute `bound = 1.0 / math.sqrt(in_features)`.
     - Sample a weight init via `w_init = t.empty(out_features, in_features).uniform_(-bound, +bound)` and wrap it: `self.weight = t.nn.Parameter(w_init)`.
     - Same for bias: `b_init = t.empty(out_features).uniform_(-bound, +bound)`, then `self.bias = t.nn.Parameter(b_init)`.
   - `forward(self, x): return x @ self.weight.T + self.bias`.
2. Return an instance of `KaimingLinear` from `ex1_kaiming_linear(...)`.

**Why this exact formula.** PyTorch's `nn.Linear.reset_parameters` calls `nn.init.kaiming_uniform_(self.weight, a=math.sqrt(5))`, which expands to `Uniform(-bound, +bound)` with `bound = sqrt(6 / ((1 + 5) * fan_in)) = 1 / sqrt(fan_in)`. The bias gets the same bound for backward-pass variance preservation.

**Use `.uniform_(low, high)` (trailing underscore = in place).** Sample into a plain tensor FIRST, then wrap in `nn.Parameter`. Calling `.uniform_` directly on a leaf Parameter with `requires_grad=True` raises `RuntimeError: a leaf Variable that requires grad is being used in an in-place operation` — you'd need a `with t.no_grad():` block to do it that way. The sample-then-wrap pattern is cleaner.

The visualization plots a histogram of the weight values against the theoretical `[-bound, +bound]` rectangle so you can confirm the distribution actually IS uniform on that interval.

In [ ]:
def ex1_kaiming_linear(in_features: int, out_features: int):
    import math
    class KaimingLinear(t.nn.Module):
        def __init__(self, in_features, out_features):
            super().__init__()
            self.in_features = in_features
            self.out_features = out_features
            bound = 1.0 / math.sqrt(in_features)
            # Sample into a plain tensor (no requires_grad) so .uniform_ is legal,
            # then wrap in nn.Parameter — same end state as initializing in place
            # on the leaf Parameter under torch.no_grad().
            w_init = t.empty(out_features, in_features).uniform_(-bound, +bound)
            self.weight = t.nn.Parameter(w_init)
            b_init = t.empty(out_features).uniform_(-bound, +bound)
            self.bias = t.nn.Parameter(b_init)
        def forward(self, x):
            return x @ self.weight.T + self.bias
    return KaimingLinear(in_features, out_features)


<details><summary>Solution</summary>

```python
def ex1_kaiming_linear(in_features: int, out_features: int):
    import math
    class KaimingLinear(t.nn.Module):
        def __init__(self, in_features, out_features):
            super().__init__()
            self.in_features = in_features
            self.out_features = out_features
            bound = 1.0 / math.sqrt(in_features)
            # Sample into a plain tensor (no requires_grad) so .uniform_ is legal,
            # then wrap in nn.Parameter — same end state as initializing in place
            # on the leaf Parameter under torch.no_grad().
            w_init = t.empty(out_features, in_features).uniform_(-bound, +bound)
            self.weight = t.nn.Parameter(w_init)
            b_init = t.empty(out_features).uniform_(-bound, +bound)
            self.bias = t.nn.Parameter(b_init)
        def forward(self, x):
            return x @ self.weight.T + self.bias
    return KaimingLinear(in_features, out_features)
```

**Why `t.empty` not `t.zeros` or `t.randn`.** `t.empty` allocates uninitialized memory (garbage values) which we immediately overwrite with `.uniform_`. Allocating with `t.zeros` would waste a write; allocating with `t.randn` would waste a different write AND set the wrong distribution before we fix it.

**Why in-place `.uniform_` not `t.nn.Parameter(t.rand(...) * scale)`.** Both work for the math, but the in-place form is what PyTorch's `reset_parameters` does, and it's the idiom every reviewer expects. Reassigning a Parameter slot via `self.weight = t.nn.Parameter(new_tensor)` works but signals to readers that you don't know about in-place init.

**The default uses `a=sqrt(5)`, which is a historical accident.** Modern He/Kaiming for ReLU uses `a=0`, which gives bound `sqrt(6 / fan_in)` — about 2.45x larger than PyTorch's default `1/sqrt(fan_in)`. For deep ReLU networks, the `a=0` variant trains faster; the `a=sqrt(5)` default exists for legacy compatibility with old PyTorch checkpoints. Almost every modern repo overrides it.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()